# Train TransUNet on Kaggle — 4-fold CV + fixed-test evaluation

Reproduces the local TransUNet experiment on a Kaggle GPU. TransUNet is
**image-only**, so it does **not** need the `data/bb_maps/` priors — you only
upload `data/splits`.

## Before you run
1. **Accelerator:** Settings → Accelerator → **GPU T4 x2** or **GPU P100**.
2. **Internet:** Settings → Internet → **On** (needed for `git clone` + `pip`).
3. **Data:** upload `data/splits.zip` (already built for you) as a **Kaggle Dataset**,
   then add it to this notebook (right panel → *Add Input*). To regenerate it,
   run `cd data && zip -r -0 splits.zip splits -x '*/yolox_coco/*'` (the exclude
   drops the YOLOX COCO symlinks, which TransUNet doesn't use and which otherwise
   double the archive). It must contain `splits/folds/fold_0..3/`, `splits/test/`,
   and `splits/class_map.txt`. Copy its mount path into `SPLITS_PATH` below.

Run the cells top to bottom. The full 4-fold run is ~2–5 h and fits one session.

## 1. Configure
Edit the values below to match your setup, then run.

In [ ]:
# === EDIT to match your uploaded Kaggle Dataset and repo ===
SPLITS_PATH = "/kaggle/input/pbl4-splits/splits"      # must contain folds/, test/, class_map.txt
REPO_URL    = "https://github.com/Huay0804/PBL4.git"  # private? use https://<TOKEN>@github.com/Huay0804/PBL4.git
REPO_DIR    = "/kaggle/working/PBL4"

## 2. Get the code and install pinned dependencies
Matches `requirements.txt` (TF 2.20 / Keras 3.13) plus the CUDA extra so the GPU
is detected. If the sanity check in Step 3 shows wrong versions or no GPU,
**Restart kernel** (Run menu) and re-run from Step 3 — cloned files persist.

In [ ]:
import os, subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

!pip install -q "tensorflow[and-cuda]==2.20.0" keras==3.13.0 numpy==2.0.2 scipy==1.15.1 scikit-image==0.25.2 pillow==12.1.0 h5py==3.12.1
print("Dependencies installed.")

## 3. Wire up data and sanity-check the environment
Links the read-only dataset to `data/splits` (the scripts resolve `data/splits/...`
relative to the repo root), then verifies TF/Keras/GPU and builds the model.

In [ ]:
import os, sys
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))

# Link data/splits -> uploaded dataset.
os.makedirs("data", exist_ok=True)
link = "data/splits"
if os.path.islink(link) or os.path.exists(link):
    if os.path.islink(link):
        os.remove(link)
os.symlink(SPLITS_PATH, link)
assert os.path.exists("data/splits/class_map.txt"), \
    f"data/splits/class_map.txt missing — check SPLITS_PATH={SPLITS_PATH}"
print("Linked data/splits ->", SPLITS_PATH, "| contents:", os.listdir("data/splits"))

import tensorflow as tf, keras
print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus or "NONE — restart kernel and re-run; confirm GPU accelerator is on")

from segmentation_models import TransUNet
_m = TransUNet(input_shape=(512, 1024, 3), classes=33, activation="softmax")
print("TransUNet params:", f"{_m.count_params():,}")
del _m

## 4. Train all 4 CV folds
Same command as local — the preset drives batch size, mixed precision,
early-stopping (patience 4) and the 5-epoch GPU-pool refresh loop.
`PBL4_GPU_DISPLAY_RESERVE_MB=0` uses the full headless GPU. If a session is cut
short, just re-run: folds merge incrementally into the CV summary.

In [ ]:
import os, subprocess
os.chdir(REPO_DIR)
os.environ["PBL4_GPU_DISPLAY_RESERVE_MB"] = "0"

for k in range(4):
    print(f"\n================== TRAIN FOLD {k} ==================", flush=True)
    rc = subprocess.run(
        f"python -u scripts/train_segmentation_cv.py --model transunet --fold {k}",
        shell=True,
    ).returncode
    if rc != 0:
        raise SystemExit(f"Fold {k} training failed (exit {rc}).")
print("\nAll folds trained.")

## 5. Evaluate each fold on the fixed test set
Writes `test_summary.json`, `test_metrics.json`, `per_class_*`, `per_position_*`,
and `per_tooth_type_*` next to each fold's checkpoint (unchanged format).

In [ ]:
import os, subprocess
os.chdir(REPO_DIR)

for k in range(4):
    print(f"\n================== EVAL FOLD {k} ==================", flush=True)
    rc = subprocess.run(
        f"python -u scripts/evaluate_final.py --model transunet --cv-fold {k}",
        shell=True,
    ).returncode
    if rc != 0:
        raise SystemExit(f"Fold {k} evaluation failed (exit {rc}).")
print("\nAll folds evaluated.")

## 6. Results
Cross-validation aggregate (validation folds) and the per-fold test-set summaries.

In [ ]:
import glob, json, os
os.chdir(REPO_DIR)

cv = "runs/cv/transunet_cv_summary.json"
if os.path.exists(cv):
    print("=== CV aggregate (validation) ===")
    print(json.dumps(json.load(open(cv)).get("aggregate", {}), indent=2))

print("\n=== Per-fold test-set summaries ===")
for p in sorted(glob.glob("runs/cv/fold_*/transunet/**/test_summary.json", recursive=True)):
    print(p)
    print(json.dumps(json.load(open(p)), indent=2))

## 7. Download results
Zips the CV outputs (checkpoints + metrics) to the notebook **Output** tab,
excluding bulky TensorBoard logs and backup checkpoints. You can also
**Save Version** to persist `/kaggle/working` automatically.

In [ ]:
import os
os.chdir(REPO_DIR)
!zip -r -q /kaggle/working/transunet_runs.zip runs/cv -x "*/logs/*" "*/.training_backup/*"
print("Wrote /kaggle/working/transunet_runs.zip — download it from the Output tab.")